# 💻 Code Generation & Analysis with Claude Sonnet 4.6

Welcome to **Contoso Financial**, a fictional financial services company modernizing a 20-year-old trading platform spread across **Java and COBOL**. The system still clears millions of transactions a day, but the CTO has greenlit a modernization push: translate legacy logic to Python, review risky code, generate comprehensive tests, and automate Azure infrastructure.

In this notebook, you'll use **Claude Sonnet 4.6 on Microsoft Foundry** like an experienced engineering partner for modernization work.

### What you'll learn
- Generate production-style code from detailed specifications
- Review and debug code for bugs, security issues, and design flaws
- Translate legacy code into modern Python
- Generate tests to build confidence before refactoring
- Use multi-turn sessions for iterative refinement
- Generate Infrastructure as Code (IaC) for Azure deployments


In [ ]:
import os

# Suppress warning messages before importing libraries that emit them.
import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", message=r".*experimental and may change or be removed.*")

from dotenv import load_dotenv

from agent_framework import Agent
from agent_framework.anthropic import AnthropicChatOptions
from agent_framework.foundry import AnthropicFoundryClient

load_dotenv(dotenv_path="../.env")

chat_client = AnthropicFoundryClient(
    model=os.environ["FOUNDRY_MODEL_DEPLOYMENT"],
    api_key=os.environ["FOUNDRY_API_KEY"],
    base_url=os.environ["FOUNDRY_ENDPOINT"],
)

agent = Agent(
    client=chat_client,
    name="code-assistant",
    instructions=(
        "You are a senior software engineer and code architect with expertise in "
        "Python, Java, COBOL, cloud infrastructure (Terraform, Bicep), and "
        "software testing. When writing code: include docstrings, type hints, "
        "error handling, and follow best practices. When reviewing code: identify "
        "bugs, security issues, performance problems, and suggest specific fixes."
    ),
    default_options=AnthropicChatOptions(max_tokens=4096, temperature=0.0),
)

setup_session = agent.create_session()
response = await agent.run(
    "Reply with exactly: Setup complete — Claude Sonnet 4.6 is ready for code generation.",
    session=setup_session,
)
print(response.text)


<br/>

---

## 🏗️ Code Generation — From Specification to Implementation

Think of Claude as a **senior developer doing pair programming** with you. Like any strong pair partner, it needs context: what are you building, what are the constraints, and what quality bar matters here?

A vague prompt is like saying, *"build me something nice"* and walking away. A detailed specification is more like handing your teammate a ticket with acceptance criteria, edge cases, and architecture notes. **Vague specs get generic code; detailed specs get production-ready code.**


In [ ]:
code_generation_session = agent.create_session()

trade_spec_prompt = """
Write a Python function called `calculate_trade_settlement` that:
- takes a trade dictionary with keys: symbol, quantity, price, trade_date, settlement_type
- calculates the settlement date based on T+1 for stocks and T+2 for bonds
- computes total value including a 0.1% commission fee
- returns a settlement dictionary with all computed fields
- includes type hints and a docstring
- handles edge cases like missing fields, invalid values, unsupported settlement types, and malformed dates

Use modern Python best practices and return only the code in a single fenced Python code block.
"""

response = await agent.run(trade_spec_prompt, session=code_generation_session)
print(response.text)


<br/>

---


## 🔍 Code Review — Finding Bugs Before They Find You

One of Claude's most useful enterprise skills is code review. It can scan for bugs, security vulnerabilities, and performance issues with the consistency of a reviewer who never gets tired, never loses context, and somehow remembers every painful postmortem.

It's a bit like having a security engineer, a senior reviewer, and a QA lead all leaning over the same pull request — minus the scheduling meeting.


In [ ]:
review_session = agent.create_session()

buggy_code = '''
def process_payment(amount, currency, account_id):
    # Convert to base currency
    rates = {"USD": 1.0, "EUR": 0.85, "GBP": 0.73}
    converted = amount * rates[currency]
    
    # Apply discount for large transactions
    if amount > 10000:
        discount = amount * 0.02
        converted = converted - discount
    
    # Format for database
    result = {
        "amount": round(converted, 2),
        "account": account_id,
        "timestamp": str(datetime.now()),
        "status": "completed"
    }
    
    # Log transaction
    log_file = open(f"/tmp/transactions_{account_id}.log", "a")
    log_file.write(str(result))
    
    return result
'''

review_prompt = f"""
Review the following Python function as if it were part of a payment pipeline at a financial services company.

For your review:
1. Identify correctness bugs, security issues, resource management problems, and missing validation.
2. Explain why each issue matters in production.
3. Suggest concrete fixes.
4. Provide a corrected version of the function.

Code to review:
{buggy_code}
"""

response = await agent.run(review_prompt, session=review_session)
print(response.text)


<br/>

---

## 🌉 Code Translation — Modernizing Legacy Systems

Contoso Financial's oldest business rules still live in legacy systems. Some logic is buried in COBOL, some in older Java services, and much of it encodes decisions the business depends on every day.

Translating that code is not just syntax conversion — it's more like using a **Rosetta Stone for software**. You want to preserve the intent, the edge cases, and the institutional knowledge, while expressing it in a modern language your current team can safely evolve.



In [ ]:
translation_session = agent.create_session()

legacy_java_code = """
// Legacy Java function
public class TradeValidator {
    public static ValidationResult validateTrade(Trade trade) {
        ValidationResult result = new ValidationResult();
        
        if (trade.getQuantity() <= 0) {
            result.addError(\"Invalid quantity\");
            return result;
        }
        
        if (trade.getPrice() < 0.01) {
            result.addError(\"Price below minimum tick\");
            return result;
        }
        
        double notional = trade.getQuantity() * trade.getPrice();
        if (notional > trade.getAccount().getRiskLimit()) {
            result.addError(\"Exceeds risk limit: \" + notional);
            return result;
        }
        
        if (isMarketClosed(trade.getExchange())) {
            result.addError(\"Market is closed\");
            return result;
        }
        
        result.setValid(true);
        result.setNotional(notional);
        return result;
    }
}
"""

translation_prompt = f"""
Translate the following legacy Java trading validation logic into modern Python.

Requirements:
- use dataclasses
- use type hints
- use Pythonic patterns
- preserve the business logic exactly
- include a short explanation of the main translation choices after the code

Java code:
{legacy_java_code}
"""

response = await agent.run(translation_prompt, session=translation_session)
print(response.text)


<br/>

---

## 🧪 Test Generation — Building a Safety Net

Before you modernize aggressively, you need tests. Think of tests as the **measuring tape before cutting fabric** — they capture expected behavior so you can refactor with confidence instead of crossed fingers.

For legacy modernization, test generation is often the bridge between *"we're scared to touch it"* and *"we can improve this safely."*


In [ ]:
test_generation_prompt = """
Generate pytest tests for the Python trade validator you just created.

Please include:
- happy path tests
- edge cases
- boundary conditions
- error cases

Return the tests in a single fenced Python code block and briefly note what each test category is protecting against.
"""

response = await agent.run(test_generation_prompt, session=translation_session)
print(response.text)


<br/>

---

## 🔁 Iterative Refinement — The Conversation Advantage

This is where sessions shine. Real development is rarely one-shot. It's more like shaping clay than stamping a coin: build the first version, inspect it, add guardrails, improve usability, then polish the design.

With multi-turn chat, you can refine code naturally:

- *Add error handling*
- *Now make it async*
- *Add logging*
- *Expose metrics*

Each turn inherits the earlier context, so the model improves the existing design instead of starting from scratch every time.


In [ ]:
order_book_session = agent.create_session()

turn_1 = await agent.run(
    "Write a Python class for a simple order book that can add buy/sell orders and match them.",
    session=order_book_session,
)
print("=== Turn 1: Initial implementation ===\n")
print(turn_1.text)

turn_2 = await agent.run(
    "Good. Now add input validation and proper error handling.",
    session=order_book_session,
)
print("\n=== Turn 2: Added validation and error handling ===\n")
print(turn_2.text)

turn_3 = await agent.run(
    "Now add a method to get the current bid-ask spread and market depth.",
    session=order_book_session,
)
print("\n=== Turn 3: Added analytics methods ===\n")
print(turn_3.text)

turn_4 = await agent.run(
    "Finally, add type hints and a docstring for each method.",
    session=order_book_session,
)
print("\n=== Turn 4: Final refined version ===\n")
print(turn_4.text)


<br/>

---

## ☁️ Infrastructure as Code — DevOps Automation

Contoso Financial is moving to Azure, and the platform team wants infrastructure defined as code rather than clicked together in a portal. IaC is the difference between a hand-built snowflake server and an assembly line blueprint.

With Claude, you can start from natural language requirements and quickly draft a **reviewable, version-controlled infrastructure template** that your cloud team can refine.


In [ ]:
iac_session = agent.create_session()

iac_prompt = """
Generate a Bicep template for an Azure deployment for a trading API with:
- a Container App for the API service
- a PostgreSQL database for trade records
- a Key Vault for secrets
- a Service Bus for async order processing
- proper networking considerations
- managed identity

Please:
- use clear parameters and outputs
- include comments where helpful
- follow secure defaults suitable for an enterprise team
- return the Bicep in a single fenced code block, followed by a short explanation of the architecture
"""

response = await agent.run(iac_prompt, session=iac_session)
print(response.text)


<br/>

---

## 🎯 Try It Yourself

Now it's your turn to treat Claude like a modernization teammate.

### Suggested exercises
1. Take a real function from your codebase and ask Claude for a review focused on bugs, security, and maintainability.
2. Translate a small snippet from one language to another and ask Claude to explain what changed semantically, not just syntactically.
3. Generate tests for an untested module before refactoring it.
4. Draft IaC from a whiteboard-style requirements list, then refine it with your platform team's standards.

### Key takeaways
- **Specific prompts produce better code.** Think like you're writing a ticket for a senior engineer.
- **Sessions unlock iteration.** You can refine code step by step instead of reprompting from zero.
- **Claude is useful across the software lifecycle.** Not just coding, but reviewing, translating, testing, and infrastructure design.
- **Foundry gives you a deployment-ready path.** The same model you test in a playground can be wired into production code through the Agent Framework.

If Contoso Financial can modernize a 20-year-old trading platform one component at a time, your team can too.
